# 15.6 - Regulatory Landscape

**Module 15 - Ethics & Responsible AI** | Netsetos GenAI Engineering

This notebook works through Regulatory Landscape hands-on: Map the regulatory regimes; Classify a system into its risk tier; Screen against the Article 5 banned list; Run the high-risk conformity checklist; Test a model against the 10^25 FLOP line; Size the penalty exposure.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

The only preamble this notebook needs. Every cell after this is pure rule-checking on hard-coded illustrative facts - no API keys, no model calls, no external data.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A one-line reproducibility note. The lesson uses fixed, hand-written values throughout rather than sampling anything, so there is nothing to seed - the comment is a placeholder marking where a seed would normally go.

**How the code works - step by step**
- A single comment line noting the lesson uses seeded/fixed values.
- No imports and no library dependencies - the whole notebook runs on plain Python built-ins.

**In one line:** nothing to install - every downstream cell is self-contained.

**What you'll see:** (no output - environment prep)

## 1 - Map the regulatory regimes

There is no single global AI law, so the first move is to know the terrain: which regimes exist and how they differ. This cell lays out the five major ones and marks the EU AI Act as the reference to design against - because it is the most comprehensive and it reaches beyond the EU.

In [ ]:
# THE REGULATORY MAP: AI is now regulated worldwide, but not uniformly. The EU AI Act is the first COMPREHENSIVE, RISK-BASED
# law and it reaches beyond the EU; other regimes are sectoral, principles-based, or content-specific. Know which apply to you.
REGIMES = [  # (regime, approach, note)
    ("EU AI Act",        "risk-based, comprehensive", "tiers by risk; extraterritorial; the reference regime"),
    ("US (federal)",     "sectoral + executive",      "no single AI law; agency rules (FTC, EEOC) + executive orders"),
    ("US (states)",      "patchwork",                 "Colorado AI Act, NYC bias-audit law, state deepfake/privacy laws"),
    ("UK",               "principles-based",          "regulator-led, pro-innovation; no single statute yet"),
    ("China",            "content + algorithm rules", "generative-AI measures, algorithm registry, labelling")]
print("The 2026 AI regulatory landscape is a patchwork of {} major regimes:".format(len(REGIMES)))
for name, approach, note in REGIMES:
    print("  {:<14} {:<26} {}".format(name, approach, note))
print()
print("There is no single global AI law. The EU AI Act is the most comprehensive and, because it is RISK-BASED and applies")
print("EXTRATERRITORIALLY, it is the one most GenAI teams design to first - so this lesson uses it as the worked example.")

# Output:
# The 2026 AI regulatory landscape is a patchwork of 5 major regimes:
#   EU AI Act      risk-based, comprehensive  tiers by risk; extraterritorial; the reference regime
#   US (federal)   sectoral + executive       no single AI law; agency rules (FTC, EEOC) + executive orders
#   US (states)    patchwork                  Colorado AI Act, NYC bias-audit law, state deepfake/privacy laws
#   UK             principles-based           regulator-led, pro-innovation; no single statute yet
#   China          content + algorithm rules  generative-AI measures, algorithm registry, labelling
#
# There is no single global AI law. The EU AI Act is the most comprehensive and, because it is RISK-BASED and applies
# EXTRATERRITORIALLY, it is the one most GenAI teams design to first - so this lesson uses it as the worked example.

**Explanation**

A plain reference table, not a computation. It stores five regulatory regimes as tuples of (regime, approach, note) and prints them in a formatted column layout. The point is orientation: the EU AI Act is risk-based and extraterritorial, so it is the one to plan for first, and the rest of the lesson zooms into it.

**How the code works - step by step**
- **`REGIMES`** - a list of five `(name, approach, note)` tuples: the EU AI Act, US federal, US states, the UK, and China.
- **The header print** - uses `len(REGIMES)` to report that the landscape is a patchwork of five regimes.
- **The `for` loop** - prints each regime left-aligned in fixed-width columns (`{:<14}`, `{:<26}`) so the approaches and notes line up.
- **The closing prints** - state the takeaway: no single global law, and the EU AI Act is the reference regime.

**In one line:** hold five regimes in a table and print them, flagging the EU AI Act as the one to design to first.

**What you'll see:** A five-row table of regimes with their approach and a note, then two lines explaining there is no single global AI law and why the EU AI Act is the worked example.

## 2 - Classify a system into its risk tier

Inside the EU AI Act everything starts with classification: the tier a system falls into decides its obligations. This cell encodes the top-down classification logic and runs four systems through it, one landing in each of the four tiers.

In [ ]:
# THE EU AI ACT RISK TIERS: the Act sorts every system into one of four tiers by the RISK it poses, and the tier decides the
# obligations. Classify a system top-down: prohibited first, then high-risk (Annex III), then limited (transparency), else minimal.
def classify(system):
    if system.get("prohibited"):
        return "PROHIBITED (Art. 5)", "banned - cannot be deployed"
    if system.get("annex_iii"):                       # e.g. employment, credit, biometrics, critical infrastructure
        return "HIGH-RISK", "allowed with strict obligations (Art. 9-15) + conformity assessment"
    if system.get("interacts_or_generates"):          # a chatbot, or generates image/audio/video/text
        return "LIMITED", "transparency only (Art. 50): tell users, label AI content"
    return "MINIMAL", "no specific obligations (most systems)"
SYSTEMS = [
    ("government social-scoring",   {"prohibited": True}),
    ("CV-screening for hiring",     {"annex_iii": True}),
    ("customer-support chatbot",    {"interacts_or_generates": True}),
    ("spam filter",                 {})]
print("Classifying four systems into the EU AI Act risk tiers:")
for name, attrs in SYSTEMS:
    tier, rule = classify(attrs)
    print("  {:<26} -> {:<18} {}".format(name, tier, rule))
print()
print("The tier is everything: it decides whether you are banned, heavily regulated, lightly regulated, or free. Most GenAI")
print("features are LIMITED (a chatbot, an image generator) and owe transparency; the compliance weight lands on HIGH-RISK uses.")

# Output:
# Classifying four systems into the EU AI Act risk tiers:
#   government social-scoring  -> PROHIBITED (Art. 5) banned - cannot be deployed
#   CV-screening for hiring    -> HIGH-RISK          allowed with strict obligations (Art. 9-15) + conformity assessment
#   customer-support chatbot   -> LIMITED            transparency only (Art. 50): tell users, label AI content
#   spam filter                -> MINIMAL            no specific obligations (most systems)
#
# The tier is everything: it decides whether you are banned, heavily regulated, lightly regulated, or free. Most GenAI
# features are LIMITED (a chatbot, an image generator) and owe transparency; the compliance weight lands on HIGH-RISK uses.

**Explanation**

The core idea in code form: a single `classify` function that checks tiers top-down and returns the first match. It reads the system's attributes in priority order - prohibited, then Annex III high-risk, then interacts/generates, else minimal - which is exactly the order the Act intends you to screen in. Read it as: the tier is decided by the first box a system ticks, from most severe to least.

**How the code works - step by step**
- **`classify(system)`** - checks flags in order: `prohibited` -> PROHIBITED, `annex_iii` -> HIGH-RISK, `interacts_or_generates` -> LIMITED, else MINIMAL; returns the tier label plus its obligation summary.
- **`SYSTEMS`** - four `(name, attrs)` examples, one per tier: government social-scoring, CV-screening, a support chatbot, a spam filter.
- **The `for` loop** - calls `classify` on each system's attributes and prints the name, the resulting tier, and its rule.
- **The closing prints** - note that the tier decides everything, and most GenAI features are LIMITED while the compliance weight lands on HIGH-RISK.

**In one line:** check the tiers top-down and return the first hit - the tier decides the obligations.

**What you'll see:** Four systems each mapped to a tier (PROHIBITED / HIGH-RISK / LIMITED / MINIMAL) with the obligation for each, then two lines noting the tier is everything and most GenAI features are limited-risk.

## 3 - Screen against the Article 5 banned list

The first and most important screen is for the prohibited tier, because it is a full stop no engineering can undo. This cell holds the seven banned categories and checks a proposed product against them - and shows a product that hits two.

In [ ]:
# PROHIBITED PRACTICES (Art. 5): some uses are banned outright, whatever the safeguards. A system that hits ANY prohibited
# category cannot be placed on the EU market. Screen against the list before anything else - a prohibited use is a full stop.
PROHIBITED = [
    "social scoring by public authorities",
    "subliminal or manipulative techniques causing harm",
    "exploiting vulnerabilities (age, disability)",
    "real-time remote biometric identification in public (law-enforcement, narrow exceptions)",
    "biometric categorization by sensitive traits",
    "emotion recognition in workplace or school",
    "untargeted scraping of faces for recognition databases"]
def screen(features):
    return [p for p in PROHIBITED if p in features]
system_features = {"emotion recognition in workplace or school",
                   "social scoring by public authorities"}     # two prohibited uses bundled into one product
hits = screen(system_features)
print("Screening a proposed 'employee wellbeing' product against the Art. 5 prohibited list:")
for p in PROHIBITED:
    print("  [{}] {}".format("X" if p in system_features else " ", p))
print()
print("result: {}".format("ALLOWED past Art. 5" if not hits else "PROHIBITED - {} banned practice(s):".format(len(hits))))
for h in hits:
    print("   - " + h)
print("A prohibited use is a hard stop, not a compliance cost: no documentation or oversight makes emotion recognition at work legal.")

# Output:
# Screening a proposed 'employee wellbeing' product against the Art. 5 prohibited list:
#   [X] social scoring by public authorities
#   [ ] subliminal or manipulative techniques causing harm
#   [ ] exploiting vulnerabilities (age, disability)
#   [ ] real-time remote biometric identification in public (law-enforcement, narrow exceptions)
#   [ ] biometric categorization by sensitive traits
#   [X] emotion recognition in workplace or school
#   [ ] untargeted scraping of faces for recognition databases
#
# result: PROHIBITED - 2 banned practice(s):
#    - social scoring by public authorities
#    - emotion recognition in workplace or school
# A prohibited use is a hard stop, not a compliance cost: no documentation or oversight makes emotion recognition at work legal.

**Explanation**

A set-membership screen, not a judgement call. `screen` intersects the product's declared features against the fixed `PROHIBITED` list and returns every match. Because a prohibited use cannot be rescued by documentation or oversight, the presence of even one hit is a hard stop - the code makes that a simple 'any matches?' test you run before anything else.

**How the code works - step by step**
- **`PROHIBITED`** - a list of the seven Article 5 banned uses (social scoring, subliminal manipulation, exploiting vulnerabilities, real-time public biometrics, biometric categorization, workplace/school emotion recognition, untargeted face-scraping).
- **`screen(features)`** - a list comprehension returning every prohibited item present in the product's feature set.
- **`system_features`** - the 'employee wellbeing' product's declared features: workplace emotion recognition plus social scoring - two banned uses.
- **The checklist loop** - prints each banned category with `[X]` or `[ ]` marking whether the product hits it.
- **The result print** - shows PROHIBITED with the count and lists the specific banned practices, then states a prohibited use is a hard stop.

**In one line:** intersect the product's features with the banned list - any hit is a full stop.

**What you'll see:** A seven-item checklist with two boxes ticked, then a PROHIBITED verdict naming the two banned practices (social scoring and workplace emotion recognition) and a line that no safeguards make them legal.

## 4 - Run the high-risk conformity checklist

High-risk is where most real compliance work lives: allowed, but only after meeting seven cumulative duties and passing a conformity assessment. This cell tracks the seven Article 9-15 obligations as a checklist and shows that one gap blocks the CE mark.

In [ ]:
# HIGH-RISK OBLIGATIONS (Art. 9-15): a high-risk system is allowed, but only if it meets seven core duties and passes a
# conformity assessment. Track them as a checklist; ONE unmet duty means it is not conformity-ready and cannot be CE-marked.
OBLIGATIONS = {
    "risk management system (Art. 9)": True,
    "data governance - quality, bias-checked (Art. 10)": True,
    "technical documentation (Art. 11)": True,
    "automatic record-keeping / logging (Art. 12)": True,
    "transparency + instructions to deployers (Art. 13)": True,
    "human oversight (Art. 14)": False,                # UNMET - no meaningful human-in-the-loop
    "accuracy, robustness, cybersecurity (Art. 15)": True}
unmet = [name for name, ok in OBLIGATIONS.items() if not ok]
print("Conformity checklist for a HIGH-RISK CV-screening system:")
for name, ok in OBLIGATIONS.items():
    print("  [{}] {}".format("x" if ok else " ", name))
print()
print("status: {}".format("CONFORMITY-READY" if not unmet else "NOT READY - {} unmet duty(ies):".format(len(unmet))))
for u in unmet:
    print("   - " + u)
print("High-risk is not banned - it is heavily governed. All seven duties plus a conformity assessment gate the CE mark; miss one and you cannot ship.")

# Output:
# Conformity checklist for a HIGH-RISK CV-screening system:
#   [x] risk management system (Art. 9)
#   [x] data governance - quality, bias-checked (Art. 10)
#   [x] technical documentation (Art. 11)
#   [x] automatic record-keeping / logging (Art. 12)
#   [x] transparency + instructions to deployers (Art. 13)
#   [ ] human oversight (Art. 14)
#   [x] accuracy, robustness, cybersecurity (Art. 15)
#
# status: NOT READY - 1 unmet duty(ies):
#    - human oversight (Art. 14)
# High-risk is not banned - it is heavily governed. All seven duties plus a conformity assessment gate the CE mark; miss one and you cannot ship.

**Explanation**

An all-or-nothing checklist evaluated in code. The seven duties are a dict of `name -> bool`; `unmet` collects every duty flagged `False`. The design mirrors the law's logic: conformity is cumulative, so a non-empty `unmet` list means 'not ready' regardless of how many of the other six are satisfied - there is no 'six of seven' partial credit.

**How the code works - step by step**
- **`OBLIGATIONS`** - a dict of the seven duties (risk management, data governance, technical documentation, record-keeping, transparency, human oversight, accuracy/robustness/cybersecurity) mapped to True/False; human oversight (Art. 14) is set `False`.
- **`unmet`** - a comprehension collecting the names of every duty that is `False`.
- **The checklist loop** - prints each duty with `[x]` or `[ ]`.
- **The status print** - reports CONFORMITY-READY only when `unmet` is empty, else NOT READY with the count and the missing duties.
- **The closing print** - stresses that all seven plus a conformity assessment gate the CE mark; miss one and you cannot ship.

**In one line:** all seven duties must be True - collect the unmet ones, and any single gap blocks the CE mark.

**What you'll see:** A seven-duty checklist with six ticked and human oversight (Art. 14) unticked, then a NOT READY status naming that single unmet duty and a line that one gap blocks the CE mark.

## 5 - Test a model against the 10^25 FLOP line

General-purpose models get a second, capability-based classification. Every GPAI owes transparency, but crossing a compute threshold triggers a whole extra tier of systemic-risk duties. This cell estimates training compute and checks two models against the bright line.

In [ ]:
# GPAI + THE SYSTEMIC-RISK FLOP THRESHOLD (Art. 51): a general-purpose model owes transparency (Art. 53); if it is powerful
# enough it also gets SYSTEMIC-RISK duties. The Act presumes systemic risk when training compute exceeds 10^25 FLOP.
THRESHOLD = 1e25                                       # FLOP; the Art. 51(2) systemic-risk presumption
def training_flop(params, tokens):
    return 6 * params * tokens                         # the standard C = 6 * N * D estimate
MODELS = [("mid-size open model", 70e9, 15e12), ("frontier model", 400e9, 15e12)]
print("Estimating training compute (C = 6 x params x tokens) against the 10^25 FLOP systemic-risk line:")
for name, N, D in MODELS:
    C = training_flop(N, D)
    systemic = C > THRESHOLD
    print("  {:<20} {:>4.0f}B params x {:>3.0f}T tokens -> {:.2e} FLOP  -> {}".format(
        name, N / 1e9, D / 1e12, C, "SYSTEMIC-RISK GPAI (Art. 55 extra duties)" if systemic else "GPAI (Art. 53 transparency only)"))
print()
print("Every GPAI provider owes a training-data summary and copyright policy (Art. 53). Above 10^25 FLOP you also owe model")
print("evaluations, adversarial testing, incident reporting, and cybersecurity (Art. 55) - the frontier-model tier of the law.")

# Output:
# Estimating training compute (C = 6 x params x tokens) against the 10^25 FLOP systemic-risk line:
#   mid-size open model    70B params x  15T tokens -> 6.30e+24 FLOP  -> GPAI (Art. 53 transparency only)
#   frontier model        400B params x  15T tokens -> 3.60e+25 FLOP  -> SYSTEMIC-RISK GPAI (Art. 55 extra duties)
#
# Every GPAI provider owes a training-data summary and copyright policy (Art. 53). Above 10^25 FLOP you also owe model
# evaluations, adversarial testing, incident reporting, and cybersecurity (Art. 55) - the frontier-model tier of the law.

**Explanation**

A one-formula computation plus a threshold comparison. `training_flop` applies the standard estimate C = 6 x params x tokens, and each model's result is compared against the 10^25 FLOP line to decide whether it is a plain GPAI or a systemic-risk GPAI. It is a calculator, not a model call - a single number decides which regulatory tier a model lands in.

**How the code works - step by step**
- **`THRESHOLD`** - `1e25` FLOP, the Article 51(2) systemic-risk presumption line.
- **`training_flop(params, tokens)`** - returns `6 * params * tokens`, the standard C = 6ND compute estimate.
- **`MODELS`** - two `(name, params, tokens)` examples: a 70B x 15T mid-size open model and a 400B x 15T frontier model.
- **The `for` loop** - computes each model's FLOP, tests `C > THRESHOLD`, and prints the params/tokens, the compute in scientific notation, and whether it is transparency-only (Art. 53) or systemic-risk (Art. 55).
- **The closing prints** - note every GPAI owes a training-data summary and copyright policy, and above the line the Article 55 duties (evals, red-teaming, incident reporting, cybersecurity) also apply.

**In one line:** estimate compute with 6 x params x tokens and compare to 10^25 to decide the GPAI tier.

**What you'll see:** Two models estimated: the 70B x 15T model at 6.30e24 FLOP (below the line, transparency only) and the 400B x 15T model at 3.60e25 FLOP (above, systemic-risk GPAI), then two lines on the Article 53 vs Article 55 duties.

## 6 - Size the penalty exposure

Regulation without teeth is advice, so the Act attaches fines built to matter to large companies. This cell computes the maximum fine for each violation tier, showing why the percentage-of-turnover construction dominates the fixed caps at scale.

In [ ]:
# PENALTIES (Art. 99): fines scale with the violation AND your size - each tier is 'the HIGHER of a fixed cap OR a percentage
# of worldwide annual turnover'. For a large company the percentage dominates, so a breach is a business risk, not a line item.
TIERS = [  # (violation, fixed cap EUR, turnover %)
    ("prohibited practice (Art. 5)",         35_000_000, 0.07),
    ("other obligation, e.g. high-risk",     15_000_000, 0.03),
    ("supplying incorrect information",       7_500_000, 0.01)]
turnover = 2_000_000_000                                # EUR 2 billion worldwide annual turnover
print("Maximum fines for a company with EUR {:.1f}B worldwide turnover (higher of cap or % of turnover):".format(turnover / 1e9))
for name, cap, pct in TIERS:
    pct_amount = turnover * pct
    fine = max(cap, pct_amount)
    driver = "turnover %" if pct_amount > cap else "fixed cap"
    print("  {:<34} max(EUR {:>4.0f}M, {:.0%} = EUR {:>4.0f}M) = EUR {:>4.0f}M  ({})".format(
        name, cap / 1e6, pct, pct_amount / 1e6, fine / 1e6, driver))
print()
print("For a EUR 2B company the turnover percentage beats every fixed cap - a prohibited-practice breach tops out at EUR 140M.")
print("(SMEs get the LOWER of the two.) Penalties this size make compliance a board-level risk, which is the point of the design.")

# Output:
# Maximum fines for a company with EUR 2.0B worldwide turnover (higher of cap or % of turnover):
#   prohibited practice (Art. 5)       max(EUR   35M, 7% = EUR  140M) = EUR  140M  (turnover %)
#   other obligation, e.g. high-risk   max(EUR   15M, 3% = EUR   60M) = EUR   60M  (turnover %)
#   supplying incorrect information    max(EUR    8M, 1% = EUR   20M) = EUR   20M  (turnover %)
#
# For a EUR 2B company the turnover percentage beats every fixed cap - a prohibited-practice breach tops out at EUR 140M.
# (SMEs get the LOWER of the two.) Penalties this size make compliance a board-level risk, which is the point of the design.

**Explanation**

A `max` computation across three fine tiers. For each violation the fine is the higher of a fixed euro cap or a percentage of worldwide turnover, and the code also records which of the two drove the result. The whole point is visible in the arithmetic: at a large turnover the percentage always beats the cap, which is what turns a fine into a board-level risk.

**How the code works - step by step**
- **`TIERS`** - three `(violation, fixed_cap, turnover_pct)` tuples: prohibited (35M / 7%), other obligations (15M / 3%), incorrect information (7.5M / 1%).
- **`turnover`** - EUR 2 billion worldwide annual turnover for the worked example.
- **The `for` loop** - computes `pct_amount = turnover * pct`, takes `fine = max(cap, pct_amount)`, and records whether the cap or the turnover percentage was the binding driver.
- **The print** - shows each tier's `max(cap, percentage)` and the resulting fine with its driver.
- **The closing prints** - note the percentage beats every cap at this size (prohibited tops out at 140M), that SMEs get the *lower* of the two, and that fines this size make compliance a board-level risk.

**In one line:** each fine is the higher of a fixed cap or a turnover percentage - and at scale the percentage always wins.

**What you'll see:** Three fine tiers computed for a EUR 2B company - 140M, 60M, and 20M, each driven by the turnover percentage - then lines noting the prohibited breach tops at 140M and SMEs get the lower of the two.

## 7 - Confirm reach and timing

Two final questions decide whether any of this binds you: does the Act apply, and when. This cell tests extraterritorial reach (Article 2) and checks each obligation's phase-in date against an illustrative 'now'.

In [ ]:
# EXTRATERRITORIALITY + THE TIMELINE GATE: the EU AI Act (Art. 2) reaches you if your system's OUTPUT is used in the EU, even
# if you are based elsewhere. And it phases in by date. Gate a release: does it apply, which duties are in force, are you ready?
def applies(provider_in_eu, output_used_in_eu):
    return provider_in_eu or output_used_in_eu         # Art. 2: EU market OR output used in the EU
IN_FORCE = {  # obligation -> the date it starts applying
    "prohibited practices (Art. 5)": "2025-02",
    "GPAI obligations (Art. 53/55)": "2025-08",
    "high-risk obligations (Annex III)": "2026-08"}
today = "2026-07"                                       # illustrative 'now'
provider_in_eu, output_used_in_eu = False, True          # a US company whose chatbot serves EU users
print("A US-based provider whose model output is used in the EU:")
print("  EU AI Act applies? {}  (Art. 2: output used in the EU is enough)".format("YES" if applies(provider_in_eu, output_used_in_eu) else "no"))
print("  in force as of {}:".format(today))
for ob, date in IN_FORCE.items():
    live = date <= today
    print("    [{}] {:<38} from {}".format("LIVE" if live else "soon", ob, date))
print()
print("Extraterritorial reach means 'we are not in the EU' is not a defence - like GDPR, the Act follows the users. And it phases")
print("in, so track the calendar: prohibited practices and GPAI duties are already live; high-risk obligations land in Aug 2026.")

# Output:
# A US-based provider whose model output is used in the EU:
#   EU AI Act applies? YES  (Art. 2: output used in the EU is enough)
#   in force as of 2026-07:
#     [LIVE] prohibited practices (Art. 5)          from 2025-02
#     [LIVE] GPAI obligations (Art. 53/55)          from 2025-08
#     [soon] high-risk obligations (Annex III)      from 2026-08
#
# Extraterritorial reach means 'we are not in the EU' is not a defence - like GDPR, the Act follows the users. And it phases
# in, so track the calendar: prohibited practices and GPAI duties are already live; high-risk obligations land in Aug 2026.

**Explanation**

A two-part gate combining a boolean applicability test with a date comparison. `applies` returns True if the provider is in the EU or - crucially - the system's output is used in the EU, mirroring GDPR's follow-the-users logic. The timeline check compares each obligation's start date against `today` as plain string dates (ISO YYYY-MM sorts correctly lexicographically) to mark what is already in force.

**How the code works - step by step**
- **`applies(provider_in_eu, output_used_in_eu)`** - returns True if either flag is set; Article 2 makes output-used-in-the-EU sufficient on its own.
- **`IN_FORCE`** - a dict mapping each obligation to its phase-in date: prohibited (2025-02), GPAI (2025-08), high-risk (2026-08).
- **`today`** - the illustrative '2026-07' now; **`provider_in_eu, output_used_in_eu`** set to `False, True` for a US company serving EU users.
- **The applicability print** - reports YES because output reaches the EU.
- **The timeline loop** - compares each date against `today` with `date <= today` and prints `[LIVE]` or `[soon]` per obligation.
- **The closing prints** - note 'we are not in the EU' is no defence and that the calendar phases in, with high-risk landing in Aug 2026.

**In one line:** applies if output reaches the EU, then flag which duties are already live by date.

**What you'll see:** A YES applicability verdict for the US provider, then the three obligations marked LIVE (prohibited, GPAI) or soon (high-risk, from 2026-08), plus two lines on GDPR-style reach and the phase-in calendar.

Seven checks, run in order, are the whole workflow: map the regimes, classify the tier, screen the Article 5 banned list, run the seven-duty high-risk checklist, test a model against the 10^25 FLOP line, size the fine, then confirm reach and timing. Each cell is a keyless paper check on illustrative facts - the tier a system lands in decides everything downstream, so you run these at design time, not launch day. This is engineering education, not legal advice; Lesson 15.7 turns this list of obligations into a running governance program (NIST AI RMF, model cards, a risk register).